# AWS Textract — Document Text Extraction (Sync)

A simple, UI-free workflow: **upload a document → extract the text**.

Building block for Generative-AI / RAG pipelines: Textract turns a PDF, scanned image, or photo into clean text. This notebook stops at clean text extraction (no RAG).

**Mode:** Synchronous (`DetectDocumentText`). File bytes go straight to the API — **no S3 bucket required**.

**Supported:** single-page `PNG`, `JPEG`, `TIFF`, and single-page `PDF`.

### Prerequisites
1. AWS CLI already authenticated (uses your default credential chain).
2. IAM permission `textract:DetectDocumentText` (+ `textract:AnalyzeDocument` for the optional cell).
3. Region: **us-east-1** (change in the config cell).

> Azure & GCP equivalents are at the **bottom of this notebook** as reference markdown.

In [ ]:
# 1. Install (run once)
# %pip install boto3 ipywidgets Pillow

In [1]:
# 2. Client + credential check
import boto3
from botocore.exceptions import BotoCoreError, ClientError

AWS_REGION = "us-east-1"
MAX_SYNC_BYTES = 10 * 1024 * 1024  # sync limit: 10 MB / 1 page

session = boto3.Session(region_name=AWS_REGION)
textract = session.client("textract")

try:
    ident = session.client("sts").get_caller_identity()
    print(f"Authenticated as: {ident['Arn']}")
except (BotoCoreError, ClientError) as e:
    raise SystemExit(f"AWS credentials missing/invalid. Run `aws configure`.\n{e}")

Authenticated as: arn:aws:iam::923307152276:user/sarath


In [2]:
# 3. Extraction helpers
def _validate(file_bytes):
    if not file_bytes:
        raise ValueError("File is empty.")
    if len(file_bytes) > MAX_SYNC_BYTES:
        raise ValueError(f"{len(file_bytes)/1048576:.1f} MB exceeds 10 MB sync limit; use async S3 API.")


def detect_text(file_bytes):
    _validate(file_bytes)
    try:
        return textract.detect_document_text(Document={"Bytes": file_bytes})
    except ClientError as e:
        code = e.response["Error"]["Code"]
        hint = {
            "UnsupportedDocumentException": "Use PNG/JPEG/TIFF or single-page PDF.",
            "InvalidParameterException": "Multi-page PDF needs the async API.",
            "AccessDeniedException": "Missing textract:DetectDocumentText.",
        }.get(code, "")
        raise RuntimeError(f"Textract failed ({code}). {hint}".strip()) from e


def extract_lines(response):
    return [
        {"text": b["Text"], "confidence": b["Confidence"]}
        for b in response.get("Blocks", [])
        if b["BlockType"] == "LINE"
    ]


def extract_plain_text(response):
    return "\n".join(l["text"] for l in extract_lines(response))


print("Helpers ready.")

Helpers ready.


In [ ]:
# 4a. Upload widget (or use the path option in 4b)
import ipywidgets as widgets
from IPython.display import display

uploader = widgets.FileUpload(accept=".png,.jpg,.jpeg,.tif,.tiff,.pdf", multiple=False, description="Upload doc")
display(uploader)
print("Pick a file, then run the next cell.")

FileUpload(value={}, accept='.png,.jpg,.jpeg,.tif,.tiff,.pdf', description='Upload doc')

Pick a file, then run the next cell.


In [ ]:
# 4b. Resolve bytes from the uploader OR a hard-coded path
from pathlib import Path

# FILE_PATH = None  # e.g. r"C:\path\to\document.png" to bypass the widget
FILE_PATH = "invoice.pdf"  # e.g. r"C:\path\to\document.png" to bypass the widget


document_bytes, document_name = None, "document"
if FILE_PATH:
    p = Path(FILE_PATH)
    document_bytes, document_name = p.read_bytes(), p.name
elif uploader.value:
    item = list(uploader.value)[0]
    document_bytes, document_name = item["content"], item["name"]

if not document_bytes:
    raise SystemExit("No document loaded. Use the widget or set FILE_PATH.")
print(f"Loaded '{document_name}' ({len(document_bytes)/1024:.1f} KB)")

In [ ]:
# 5. Extract the text
response = detect_text(document_bytes)
lines = extract_lines(response)
text = extract_plain_text(response)

print(f"Detected {len(lines)} lines\n" + "=" * 50)
print(text)

In [ ]:
# 6. Per-line confidence
for ln in lines:
    flag = "  <-- low" if ln["confidence"] < 90 else ""
    print(f"[{ln['confidence']:5.1f}%] {ln['text']}{flag}")

In [ ]:
# 7. Save extracted text
out_path = Path(f"{Path(document_name).stem}_extracted.txt")
out_path.write_text(text, encoding="utf-8")
print(f"Saved -> {out_path.resolve()}")

In [ ]:
# 8. (Optional) Find a specific piece of text
KEYWORD = "total"
hits = [ln for ln in lines if KEYWORD.lower() in ln["text"].lower()]
print(f"{len(hits)} line(s) containing '{KEYWORD}':")
for ln in hits:
    print("  -", ln["text"])

In [ ]:
# 9. (Optional) Forms key-value pairs via AnalyzeDocument
def analyze_forms(file_bytes):
    _validate(file_bytes)
    resp = textract.analyze_document(Document={"Bytes": file_bytes}, FeatureTypes=["FORMS"])
    blocks = {b["Id"]: b for b in resp["Blocks"]}

    def words_for(block):
        out = []
        for rel in block.get("Relationships", []):
            if rel["Type"] == "CHILD":
                for cid in rel["Ids"]:
                    c = blocks[cid]
                    if c["BlockType"] == "WORD":
                        out.append(c["Text"])
                    elif c["BlockType"] == "SELECTION_ELEMENT" and c["SelectionStatus"] == "SELECTED":
                        out.append("[X]")
        return " ".join(out)

    pairs = []
    for b in resp["Blocks"]:
        if b["BlockType"] == "KEY_VALUE_SET" and "KEY" in b.get("EntityTypes", []):
            value = ""
            for rel in b.get("Relationships", []):
                if rel["Type"] == "VALUE":
                    value = words_for(blocks[rel["Ids"][0]])
            pairs.append((words_for(b), value))
    return pairs


try:
    for k, v in analyze_forms(document_bytes):
        print(f"{k!r:40} -> {v!r}")
except ClientError as e:
    print("AnalyzeDocument skipped:", e.response["Error"]["Code"])

---
**Multi-page PDFs (async):** sync caps at 1 page / 10 MB. For multi-page PDFs, upload to S3 and call `start_document_text_detection`, then poll `get_document_text_detection`. Left out to keep this demo simple — ask to add it.

---
# Same Extraction on Azure & Google Cloud (reference)

Everything above uses **AWS Textract**. Below is the **equivalent Python** for the other two clouds so you can compare all three.

These are **reference snippets only** — they need their own SDKs/credentials and won't run in this AWS notebook. Copy them into your own script.

| | AWS Textract | Azure AI Document Intelligence | Google Cloud Document AI |
|---|---|---|---|
| Python SDK | `boto3` | `azure-ai-documentintelligence` | `google-cloud-documentai` |
| Auth | AWS CLI / IAM | API key or Entra ID | service-account JSON / ADC |
| Plain-text call | `detect_document_text` | model `prebuilt-read` | a `processor` (OCR) |
| Forms/tables | `analyze_document` (`FORMS`,`TABLES`) | `prebuilt-layout` / `prebuilt-document` | Form Parser processor |
| Text field in result | `Blocks[].Text` | `result.content` | `document.text` |
| Multi-page PDF | async + S3 | native | online (small) or batch + GCS |

All three: send file bytes → get text + layout + confidence. Concept identical, plumbing differs.

## Azure AI Document Intelligence

Formerly *Azure Form Recognizer*. `prebuilt-read` ≈ Textract's `DetectDocumentText` (pure OCR).

**Setup**
```bash
pip install azure-ai-documentintelligence
```
Create a *Document Intelligence* resource in the Azure Portal, then set its endpoint + key:
```powershell
# Windows PowerShell
$env:AZURE_DI_ENDPOINT = "https://<your-resource>.cognitiveservices.azure.com/"
$env:AZURE_DI_KEY = "<your-key>"
```

**Extract text from a local file**
```python
import os
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient

client = DocumentIntelligenceClient(
    endpoint=os.environ["AZURE_DI_ENDPOINT"],
    credential=AzureKeyCredential(os.environ["AZURE_DI_KEY"]),
)

def extract_text(path: str) -> str:
    with open(path, "rb") as f:
        poller = client.begin_analyze_document("prebuilt-read", body=f)
    result = poller.result()        # waits for the async job
    return result.content           # full document text

def extract_lines(path: str):
    with open(path, "rb") as f:
        result = client.begin_analyze_document("prebuilt-read", body=f).result()
    for page in result.pages:
        for line in page.lines:
            yield line.content      # one detected line

print(extract_text("document.pdf"))
```

**Notes**
- Handles multi-page PDFs natively — no async/S3 dance like Textract.
- For key-value/tables, swap `"prebuilt-read"` → `"prebuilt-layout"` or `"prebuilt-document"`.
- In production prefer Entra ID: `DefaultAzureCredential()` from `azure-identity` instead of `AzureKeyCredential`.

## Google Cloud Document AI

You first create a **Processor** (e.g. a *Document OCR* processor) in the GCP console; its `PROCESSOR_ID` is what you call.

**Setup**
```bash
pip install google-cloud-documentai
```
1. Enable the **Document AI API** in your project.
2. Create a processor → note its **ID**, the **project ID**, and **location** (`us` or `eu`).
3. Authenticate:
```bash
gcloud auth application-default login
# or: export GOOGLE_APPLICATION_CREDENTIALS="/path/to/key.json"
```

**Extract text from a local file**
```python
from google.cloud import documentai

PROJECT_ID   = "your-gcp-project"
LOCATION     = "us"               # "us" or "eu"
PROCESSOR_ID = "your-processor-id"

client = documentai.DocumentProcessorServiceClient(
    client_options={"api_endpoint": f"{LOCATION}-documentai.googleapis.com"}
)
NAME = client.processor_path(PROJECT_ID, LOCATION, PROCESSOR_ID)

def extract_text(path: str, mime_type: str = "application/pdf") -> str:
    with open(path, "rb") as f:
        raw = documentai.RawDocument(content=f.read(), mime_type=mime_type)
    request = documentai.ProcessRequest(name=NAME, raw_document=raw)
    document = client.process_document(request=request).document
    return document.text            # full document text

def extract_lines(path: str, mime_type: str = "application/pdf"):
    with open(path, "rb") as f:
        raw = documentai.RawDocument(content=f.read(), mime_type=mime_type)
    document = client.process_document(
        request=documentai.ProcessRequest(name=NAME, raw_document=raw)
    ).document
    full = document.text
    for page in document.pages:     # lines store offsets into document.text
        for line in page.lines:
            seg = line.layout.text_anchor.text_segments[0]
            yield full[int(seg.start_index):int(seg.end_index)]

print(extract_text("document.pdf"))
```

**Notes**
- `mime_type` must match the file: `application/pdf`, `image/png`, `image/jpeg`, `image/tiff`.
- Online `process_document` suits small docs; for large/many-page PDFs use `batch_process_documents` with **Google Cloud Storage** — analogous to Textract's async S3 flow.
- Text lives in `document.text`; lines/tables/fields store **offsets** into that string, not copies.